# Translate MEXT Education Data to English

This notebook reads the Japanese `mext.教育コンテンツ` Delta table and uses
Fabric AI functions (`df.ai.translate`) to translate the column data from
Japanese to English, then writes the result to `mext.education_content`.

- Requires: Fabric Runtime 1.3+ and AI functions enabled on the capacity
- Source table: `mext.教育コンテンツ` (Japanese)
- Target table: `mext.education_content` (English)

In [ ]:
# Read the Japanese source table
df_jp = spark.table('mext.教育コンテンツ')
print(f'Source table mext.教育コンテンツ has {df_jp.count()} rows')
df_jp.printSchema()

In [ ]:
# Define columns to translate and their English names
# Columns with Japanese text that need AI translation
translate_cols = {
    '教材_名称': 'material_name',
    '教材_言語': 'material_language',
    '教材_キーワード': 'material_keywords',
    '教材_形式': 'material_format',
    '教材_対象者': 'material_target_audience',
    '教材_教科等': 'material_subject',
    '教材_分野_科目': 'material_field',
    '教材_対象学年': 'material_target_grade',
    '教材_コンテンツ形式': 'material_content_type',
    '教材_価格_区分': 'material_price_category',
    '教材_ライセンス': 'material_license',
    '教材_状態': 'material_status',
    '教材_公開者': 'material_publisher',
}

# Columns that just need renaming (no translation)
rename_only = {
    '教材_ID': 'material_id',
    '教材_ＵＲＬ': 'material_url',
}

print(f'{len(translate_cols)} columns to translate')
print(f'{len(rename_only)} columns to rename only')

In [ ]:
# Translate each Japanese text column to English using Fabric AI
df = df_jp

for jp_col, en_col in translate_cols.items():
    print(f'Translating {jp_col} → {en_col} ...')
    df = df.ai.translate(
        to_lang='english',
        input_col=jp_col,
        output_col=en_col
    )

print('All columns translated')

In [ ]:
# Rename non-translatable columns and select final English columns
from pyspark.sql.functions import col

# Build the final column list in a logical order
df_en = df.select(
    col('教材_ID').alias('material_id'),
    col('material_name'),
    col('material_language'),
    col('material_keywords'),
    col('material_format'),
    col('material_target_audience'),
    col('material_subject'),
    col('material_field'),
    col('material_target_grade'),
    col('material_content_type'),
    col('教材_ＵＲＬ').alias('material_url'),
    col('material_price_category'),
    col('material_license'),
    col('material_status'),
    col('material_publisher'),
)

print('English DataFrame schema:')
df_en.printSchema()
df_en.show(5, truncate=40)

In [ ]:
# Write English table to Delta
df_en.write.format('delta').mode('overwrite').saveAsTable('mext.education_content')

count = spark.table('mext.education_content').count()
print(f'Table mext.education_content has {count} rows')

# Show subject breakdown in English
print('\nRows by subject:')
spark.sql("""
    SELECT material_subject AS subject, COUNT(*) AS count
    FROM mext.education_content
    GROUP BY material_subject
    ORDER BY count DESC
""").show(truncate=False)